# Mamba: selezione e scan

Il codice del capitolo [«Mamba: selezione e scan»](https://book.paithon.it/main/StateSpaceModel/mamba.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Mamba: selezione e scan

[Leggi la pagina](https://book.paithon.it/main/StateSpaceModel/mamba.html)


### Lo scan hardware-aware


In [ ]:
import torch

# SSM selettivo, un canale: stato h di dimensione N.
# I parametri B, C, delta dipendono dal token (indice t); A e' fisso.
def ssm_selettivo(x, A, B, C, delta):
    # x: (L,)   input del canale
    # A: (N,)   diagonale fissa (valori negativi, per stabilita')
    # B, C: (L, N)  generati da x, cambiano a ogni passo
    # delta: (L,)   passo di discretizzazione, generato da x
    L, N = B.shape
    h = torch.zeros(N, dtype=x.dtype, device=x.device)
    y = torch.empty_like(x)
    for t in range(L):
        A_bar = torch.exp(delta[t] * A)   # A-bar_t = exp(delta_t A), diagonale
        B_bar = delta[t] * B[t]           # discretizzazione semplificata di B
        h = A_bar * h + B_bar * x[t]      # h_t = A-bar_t h_{t-1} + B-bar_t x_t
        y[t] = torch.dot(C[t], h)         # y_t = C_t . h_t
    return y

In [ ]:
torch.manual_seed(0)
L, N = 12, 4
x = torch.randn(L, dtype=torch.float64)
A = -torch.rand(N, dtype=torch.float64) - 0.5      # autovalori negativi
B_fisso = torch.randn(N, dtype=torch.float64)
C_fisso = torch.randn(N, dtype=torch.float64)
delta = torch.full((L,), 0.4, dtype=torch.float64)

# stessi parametri a ogni passo: il sistema e' invariante nel tempo (LTI)
y_ric = ssm_selettivo(x, A, B_fisso.repeat(L, 1), C_fisso.repeat(L, 1), delta)

# il kernel K_j = C A-bar^j B-bar: quanto pesa ancora un ingresso di j passi fa
A_bar = torch.exp(0.4 * A)
B_bar = 0.4 * B_fisso
K = torch.stack([(C_fisso * A_bar**j * B_bar).sum() for j in range(L)])

# la convoluzione causale con quel kernel, scritta a mano
y_conv = torch.stack([(K[: t + 1] * torch.flip(x[: t + 1], (0,))).sum()
                      for t in range(L)])

print("ricorrenza vs convoluzione, scarto massimo:",
      (y_ric - y_conv).abs().max().item())

In [ ]:
def scan_parallelo(a, b):
    """Ricorrenza h_t = a_t h_{t-1} + b_t svolta a raddoppio.

    Ogni passo e' la coppia (a_t, b_t), e comporne due da'
    (a1, b1) . (a2, b2) = (a2 a1, a2 b1 + b2): l'operazione e'
    associativa, quindi i passi si possono raggruppare a piacere.
    """
    a, b = a.clone(), b.clone()
    salto = 1
    while salto < a.shape[0]:
        a_prec, b_prec = a[:-salto].clone(), b[:-salto].clone()
        b[salto:] = a[salto:] * b_prec + b[salto:]
        a[salto:] = a[salto:] * a_prec
        salto *= 2          # 1, 2, 4, 8, ...: log L giri invece di L
    return b                # b_t contiene ora h_t

# parametri che cambiano a ogni passo: il sistema e' selettivo
B = torch.randn(L, N, dtype=torch.float64)
C = torch.randn(L, N, dtype=torch.float64)
delta = torch.rand(L, dtype=torch.float64) * 0.5 + 0.1
y_ciclo = ssm_selettivo(x, A, B, C, delta)

A_bar = torch.exp(delta[:, None] * A)          # (L, N)
B_bar = delta[:, None] * B * x[:, None]        # (L, N)
H = scan_parallelo(A_bar, B_bar)               # tutti gli stati in una volta
y_scan = (C * H).sum(dim=1)

print("ciclo vs scan parallelo, scarto massimo:",
      (y_ciclo - y_scan).abs().max().item())